# PRACTICE FILE — Encoding & Scikit-learn Pipelines
### Dataset: IBM Telco Customer Churn

**Rules of this notebook:**
- Every section has a **short brief** (3-5 lines) reminding you of the concept. That is NOT the lesson — you already have the full guide (`Encoding_Pipelines_Guide.docx`) and the worked example (`Telco_Churn_Encoding_Pipelines.ipynb`). Go back to those if a brief isn't enough.
- Every brief is followed by **Tasks**. You write the code in the empty cell below each task. Cells marked `# TODO` are yours to complete — do not skip them, do not peek at the worked-example notebook while attempting a task for the first time.
- Tasks marked **(Challenge)** are harder and optional-but-expected if you want to actually be good at this.
- Some cells contain a **Self-check** — run it after your task. If it prints an error, your code is wrong. Fix it before moving on. A self-check passing does not guarantee a perfect answer, but a failing one guarantees a broken one.
- No solutions are provided in this file. That is the point of a practice file.

**Setup below is given to you — do not skip running it.**


## 0. Setup (Given — just run these)

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

print("Setup complete.")

Setup complete.


In [2]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df = df.drop(columns=['customerID'])

print("Shape:", df.shape)
df.head()

Shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Encoding — Brief

Machine learning models only understand numbers. Encoding converts
categorical (text) columns into numeric form. **Nominal** data (no
order, e.g. `gender`) usually gets **One-Hot Encoding**. **Ordinal**
data (real order, e.g. `Contract`) gets **Ordinal Encoding** with a
manually defined order. High-cardinality columns often use
**Frequency** or **Target Encoding**. Label Encoding is best kept for
the target column or tree-based models.


### Task 1.1 — Clean `TotalCharges`
`TotalCharges` is loaded as text (`object`) because a few rows contain
blank strings instead of numbers.
- Convert `TotalCharges` to a numeric dtype.
- Print how many missing values exist after the conversion.
- Do **not** drop or fill them yet — that comes later with a proper `SimpleImputer`.


In [3]:
# df.info()
df.TotalCharges.isna().sum()

np.int64(0)

In [4]:
# TODO: convert df['TotalCharges'] to numeric, coercing errors to NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
# TODO: print the number of missing values in TotalCharges after conversion
print(df.TotalCharges.isna().sum())
df[df.TotalCharges.isna()]


11


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


### Task 1.2 — Label Encode the target column
Encode `Churn` ('Yes'/'No') into a numeric column called `Churn_encoded`
using `LabelEncoder`. Print which string mapped to 0 and which to 1.


In [5]:
# TODO: create Churn_encoded using LabelEncoder
df.Churn.unique()
le = LabelEncoder()
df["Churn_encoded"] = le.fit_transform(df.Churn)


In [6]:
df[["Churn", "Churn_encoded"]]

,Churn,Churn_encoded
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1
...,...,...
7038,No,0
7039,No,0
7040,No,0
7041,Yes,1


In [7]:
# TODO: print the mapping (which class -> which integer)
print(dict(zip(le.classes_, le.transform(le.classes_))))
df.head()

{'No': np.int64(0), 'Yes': np.int64(1)}


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_encoded
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


### Task 1.3 — Train/Test split BEFORE further encoding
Split the data into `X_train`, `X_test`, `y_train`, `y_test`
(80/20, `random_state=42`, `stratify` on the target). Use `Churn_encoded`
as `y`, and drop both `Churn` and `Churn_encoded` from `X`.

**Why does the split have to happen before you fit any Frequency/Target
encoder?** Answer this in a one-line comment in your code cell.


In [8]:
# The split has to come first so frequency and target encodings are fit on training rows only.
X = df.drop(columns=["Churn", "Churn_encoded"])
y = df["Churn_encoded"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

(5634, 19) (1409, 19)


### Self-check 1.3

In [9]:
# Self-check — do not edit, just run
try:
    assert X_train.shape[0] == 5634 and X_test.shape[0] == 1409
    assert 'Churn' not in X_train.columns and 'Churn_encoded' not in X_train.columns
    print("Self-check passed.")
except NameError:
    print("X_train / X_test not defined yet — complete Task 1.3 first.")
except AssertionError:
    print("Shapes or columns look wrong — check your split and dropped columns.")

Self-check passed.


### Task 1.4 — One-Hot Encode two nominal columns
Using `OneHotEncoder` (with `drop='first'` and `handle_unknown='ignore'`),
one-hot encode the `gender` and `InternetService` columns of `X_train`.
Print the resulting shape and the generated column names
(`get_feature_names_out`).


In [10]:
ohe = OneHotEncoder(drop="first", handle_unknown="ignore")
encoded = ohe.fit_transform(X_train[["gender", "InternetService"]])
print(encoded.shape)
print(ohe.get_feature_names_out())

(5634, 3)
['gender_Male' 'InternetService_Fiber optic' 'InternetService_No']


### Task 1.5 — Ordinal Encode `Contract`
`Contract` has a real order: `Month-to-month` < `One year` < `Two year`.
Use `OrdinalEncoder` with a manually specified `categories` order (do
**not** let it guess alphabetically) to create `Contract_ordinal`.
Print the unique (`Contract`, `Contract_ordinal`) pairs sorted by the
encoded value, to prove the order is correct.


In [11]:
ordinal_encoder = OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]])
X_train_ordinal = X_train.copy()
X_train_ordinal["Contract_ordinal"] = ordinal_encoder.fit_transform(X_train_ordinal[["Contract"]])
print(X_train_ordinal[["Contract", "Contract_ordinal"]].drop_duplicates().sort_values("Contract_ordinal"))

            Contract  Contract_ordinal
3738  Month-to-month               0.0
2257        One year               1.0
4860        Two year               2.0

### Task 1.6 — Frequency Encode `PaymentMethod`
Implement Frequency Encoding **manually with pandas** (no sklearn class
for this one). Use only `X_train` to compute the frequency map — this
matters, think about why. Apply the same map to a copy of `X_train`
as a new column `PaymentMethod_freq`.


In [12]:
freq_map = X_train["PaymentMethod"].value_counts(normalize=True)
X_train_freq = X_train.copy()
X_train_freq["PaymentMethod_freq"] = X_train_freq["PaymentMethod"].map(freq_map)
print(freq_map.sort_values(ascending=False))

PaymentMethod
Electronic check             0.335641
Mailed check                 0.228257
Bank transfer (automatic)    0.220802
Credit card (automatic)      0.215300
Name: proportion, dtype: float64


### Task 1.7 (Challenge) — Target Encode `PaymentMethod`
Compute the mean of `y_train` grouped by `PaymentMethod`, using
`X_train`/`y_train` ONLY. Create `PaymentMethod_target` on a copy of
`X_train`. Then explain in a comment: what would go wrong if you
computed this mean using the FULL dataset (train + test) instead?


In [13]:
combined = X_train[["PaymentMethod"]].copy()
combined["y"] = y_train
target_map = combined.groupby("PaymentMethod")["y"].mean()
X_train_target = X_train.copy()
X_train_target["PaymentMethod_target"] = X_train_target["PaymentMethod"].map(target_map)
print(target_map.sort_values(ascending=False))
# Using the full dataset would leak test-set labels into the feature, so the score would look better than it is.

PaymentMethod
Electronic check             0.457430
Mailed check                 0.192846
Bank transfer (automatic)    0.161576
Credit card (automatic)      0.149217
Name: y, dtype: float64


## 2. ColumnTransformer — Brief

Real datasets mix numeric and categorical columns that need different
treatment. `ColumnTransformer` routes each group of columns to its own
transformer (e.g. `StandardScaler` for numeric, `OneHotEncoder` for
categorical) and concatenates the results into one feature matrix — in
a single object, safe to reuse identically on train, test, and
production data.


### Task 2.1 — Define your column groups
Create two Python lists:
- `numeric_features`: all numeric columns (`tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`)
- `categorical_features`: every remaining column in `X_train` that is NOT in `numeric_features`

Do not hardcode the categorical list by hand — derive it programmatically
from `X_train.columns`.


In [14]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical_features = [c for c in X_train.columns if c not in numeric_features]
print(numeric_features)
print(categorical_features)

['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


### Task 2.2 — Build the ColumnTransformer
Build a `ColumnTransformer` named `preprocessor` with:
- a numeric branch: `SimpleImputer(strategy='median')` -> `StandardScaler()`
- a categorical branch: `SimpleImputer(strategy='most_frequent')` -> `OneHotEncoder(handle_unknown='ignore')`

Use `Pipeline` for each branch (two mini-pipelines inside the
ColumnTransformer — this is a very common real-world pattern).


In [15]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

### Task 2.3 — Fit and inspect
`fit_transform` your `preprocessor` on `X_train`. Print the shape before
and after transformation, and explain in one comment why the number of
columns changed.


In [16]:
X_train_transformed = preprocessor.fit_transform(X_train)
print(X_train.shape)
print(X_train_transformed.shape)
# One-hot encoding turns each category into its own column, so the matrix gets wider.

(5634, 19)
(5634, 45)


### Self-check 2.3

In [17]:
try:
    assert X_train_transformed.shape[0] == X_train.shape[0]
    assert X_train_transformed.shape[1] > X_train.shape[1]
    print("Self-check passed.")
except NameError:
    print("preprocessor / X_train_transformed not defined yet — complete Task 2.2-2.3 first.")
except AssertionError:
    print("Shape looks wrong — the transformed matrix should have MORE columns than the raw one (because of One-Hot Encoding).")

Self-check passed.


### Task 2.4 (Challenge) — `remainder` parameter
Rebuild the same `ColumnTransformer` but this time deliberately leave
ONE categorical column out of `categorical_features` when you define
the transformer list. Run it once with `remainder='drop'` and once with
`remainder='passthrough'`. Print the resulting shapes for both and
explain the difference in one comment.


In [18]:
reduced_categorical = categorical_features[:-1]

preprocessor_drop = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, reduced_categorical),
    ],
    remainder="drop",
)
preprocessor_pass = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, reduced_categorical),
    ],
    remainder="passthrough",
)
dropped = preprocessor_drop.fit_transform(X_train)
passed = preprocessor_pass.fit_transform(X_train)
print(dropped.shape)
print(passed.shape)
# drop throws away the column left out of the transformer; passthrough keeps it, so that result has one more column.

(5634, 41)
(5634, 42)


## 3. Pipeline — Brief

A `Pipeline` chains preprocessing steps and a final estimator into ONE
object with a single `.fit()` / `.predict()`. This prevents data
leakage, makes cross-validation correct, and is what you'd actually
ship to production.


### Task 3.1 — Build a full Pipeline
Combine your `preprocessor` (from Task 2.2, using `remainder='drop'`
version) with a `LogisticRegression(max_iter=1000, random_state=42)`
into a `Pipeline` named `log_reg_pipeline`.


In [19]:
log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

### Task 3.2 — Fit, predict, evaluate
Fit `log_reg_pipeline` on the training data, predict on the test data,
and print the accuracy and full `classification_report`.


In [20]:
log_reg_pipeline.fit(X_train, y_train)
y_pred = log_reg_pipeline.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.8055358410220014
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



### Self-check 3.2

In [21]:
try:
    acc = accuracy_score(y_test, log_reg_pipeline.predict(X_test))
    assert acc > 0.70
    print(f"Self-check passed. Accuracy = {acc:.4f}")
except NameError:
    print("log_reg_pipeline not fitted yet — complete Task 3.1-3.2 first.")
except AssertionError:
    print(f"Accuracy ({acc:.4f}) looks too low — double check your preprocessing and target.")

Self-check passed. Accuracy = 0.8055


### Task 3.3 — Swap the model
Copy your pipeline and swap only the final estimator for a
`RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)`.
You should only need to change ONE line. Fit and print its test accuracy.


In [22]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)),
])
rf_pipeline.fit(X_train, y_train)
print(accuracy_score(y_test, rf_pipeline.predict(X_test)))

0.8026969481902059


### Task 3.4 (Challenge) — GridSearchCV
Use `GridSearchCV` on `log_reg_pipeline` to search over at least 4
different values of `classifier__C`, with `cv=5`. Print the best
parameters and the best cross-validated accuracy.


In [23]:
param_grid = {"classifier__C": [0.01, 0.1, 1, 10]}
grid_search = GridSearchCV(log_reg_pipeline, param_grid, cv=5)
grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

{'classifier__C': 10}
0.8045768249380222


## 4. make_pipeline vs Pipeline — Brief

`make_pipeline()` builds the exact same kind of object as `Pipeline()`,
but auto-generates step names from each object's lowercase class name
instead of letting you choose them. Behavior is identical; only the
step names (and therefore `GridSearchCV` parameter names) differ.


### Task 4.1 — Rebuild with make_pipeline
Rebuild the exact same Logistic Regression pipeline from Task 3.1, this
time using `make_pipeline()` instead of `Pipeline()`. Call it `pipe_mp`.


In [24]:
pipe_mp = make_pipeline(preprocessor, LogisticRegression(max_iter=1000, random_state=42))

### Task 4.2 — Compare step names
Fit `pipe_mp` on the training data. Print `.named_steps.keys()` for
BOTH `log_reg_pipeline` and `pipe_mp` side by side.


In [25]:
pipe_mp.fit(X_train, y_train)
print(list(log_reg_pipeline.named_steps.keys()))
print(list(pipe_mp.named_steps.keys()))

['preprocessor', 'classifier']
['columntransformer', 'logisticregression']


### Task 4.3 — Verify identical predictions
Prove that `log_reg_pipeline` and `pipe_mp` produce IDENTICAL
predictions on `X_test` using `np.array_equal`.


In [26]:
print(np.array_equal(log_reg_pipeline.predict(X_test), pipe_mp.predict(X_test)))

True


### Self-check 4.3

In [27]:
try:
    same = np.array_equal(log_reg_pipeline.predict(X_test), pipe_mp.predict(X_test))
    assert same
    print("Self-check passed — predictions are identical, as expected.")
except NameError:
    print("pipe_mp not fitted yet — complete Task 4.1-4.2 first.")
except AssertionError:
    print("Predictions differ — something in pipe_mp is not built the same way as log_reg_pipeline.")

Self-check passed — predictions are identical, as expected.


### Task 4.4 (Challenge) — GridSearchCV with make_pipeline
Repeat Task 3.4's grid search, but this time on `pipe_mp`. You will need
to figure out the correct auto-generated parameter name yourself
(hint: print `pipe_mp.get_params().keys()` and look for the one ending
in `__C`).


In [28]:
c_params = [key for key in pipe_mp.get_params().keys() if key.endswith("__C")]
print(c_params)
param_grid = {c_params[0]: [0.01, 0.1, 1, 10]}
grid_mp = GridSearchCV(pipe_mp, param_grid, cv=5)
grid_mp.fit(X_train, y_train)
print(grid_mp.best_params_)
print(grid_mp.best_score_)

['logisticregression__C']


{'logisticregression__C': 10}
0.8045768249380222


## 5. Final Mini-Project (Challenge)

Put everything together. No more step-by-step hand-holding.

**Your task:**
1. Build the best pipeline you can for predicting `Churn` on this
   dataset (you choose: Logistic Regression, Random Forest, or anything
   else from sklearn you're comfortable with).
2. Tune at least one hyperparameter with `GridSearchCV` or
   `cross_val_score`.
3. Report final **test set** accuracy AND a full `classification_report`.
4. Save your final trained pipeline to a file called
   `my_churn_pipeline.pkl` using `joblib`.
5. In a markdown cell below your code, write 3-4 sentences explaining:
   - Which encoding choices you made and why.
   - Whether `ColumnTransformer` + `Pipeline` made this easier than doing
     it manually, and specifically how.


In [29]:
import joblib

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])
final_grid = GridSearchCV(final_pipeline, {"classifier__C": [0.1, 1, 10]}, cv=5)
final_grid.fit(X_train, y_train)
final_model = final_grid.best_estimator_
final_pred = final_model.predict(X_test)
print(final_grid.best_params_)
print(accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred))
joblib.dump(final_model, "my_churn_pipeline.pkl")

{'classifier__C': 10}
0.8055358410220014
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



['my_churn_pipeline.pkl']

### Your write-up

Nominal columns were one-hot encoded because they have no real order, numeric columns were median-imputed and scaled, and the target was label encoded before the split. Logistic regression was tuned on C with 5-fold cross-validation, then scored once on the held-out test set. Putting the steps in a ColumnTransformer and a Pipeline kept the fit inside the training data during cross-validation, so the same preprocessing is applied to the test set without leaking target information from it.

---
### Done?
Go back to `Encoding_Pipelines_Guide.docx` Section 5 (Cheat Sheet) and
`Telco_Churn_Encoding_Pipelines.ipynb` and compare your answers. If
something doesn't match and you don't understand why — that's the part
you actually need to study, not the part you got right.
